# VAZHI DAPT v2.1 — Tamil Language Adaptation Training

**Key changes from v2.0:**
1. **39.5M tokens** (was 4.8M — insufficient for language acquisition)
2. **Vanilla Qwen3-0.6B** base (fresh start — enough tokens for full DAPT)
3. **10.9% chat replay** (was 1.4% — critical for instruction preservation)
4. **5-source mix** — Wiki_Chat 70%, Sadhguru 10%, Chat replay 11%, WikiHow 8%, Classical 1%
5. **Multi-epoch with interim eval gates** — built into notebook structure
6. **Fresh cosine LR per epoch** — critical fix from v2.0

```
Step 1 (DONE): Data Prep — Vazhi_DAPT_Data_v2_1.ipynb
  -> CryptoYogi/vazhi-dapt-tamil-v2_1 (38,580 blocks x 1024 = 39.5M tokens)

Step 2 (THIS NOTEBOOK): DAPT Training — GPU (Colab Pro)
  -> Input:  CryptoYogi/vazhi-dapt-tamil-v2_1 + Qwen/Qwen3-0.6B
  -> Output: CryptoYogi/vazhi-dapt-v2_1
```

**Runtime:** ~2-3 hours per epoch on T4, ~1 hour on A100

In [ ]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.45.0,<5.0.0" \
  "datasets>=2.21.0" \
  "peft>=0.13.0" \
  "accelerate>=0.34.0" \
  "huggingface_hub>=0.24.7"

import torch
print(f"\u2705 Dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB")

In [ ]:
# Cell 2 — Configuration

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import re
import random
import glob
import gc
import torch
import numpy as np
from dataclasses import dataclass
from collections import Counter
from datasets import load_dataset
from huggingface_hub import login, HfApi

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainerCallback, Trainer, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === KEY CONFIG ===
BASE_MODEL = "Qwen/Qwen3-0.6B"              # Vanilla instruct (fresh start)
DAPT_DATASET = "CryptoYogi/vazhi-dapt-tamil-v2_1"  # 39.5M tokens
OUTPUT_MODEL = "CryptoYogi/vazhi-dapt-v2_1"        # Merged output
OUTPUT_ADAPTER = "CryptoYogi/vazhi-dapt-v2_1-lora"  # Adapter backup

# Training config
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 1e-5         # Conservative (lesson from v1.1)
BATCH_SIZE = 4               # Per-device, adjust for VRAM (4 for T4, 8 for A100)
GRADIENT_ACCUMULATION = 2    # Effective batch = 4 * 2 = 8
WARMUP_RATIO = 0.05

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Qwen3 instruct <think> tokens to suppress during generation
THINK_TOKEN_IDS = [151667, 151668]  # <think>, </think>

# GPU check
n_gpus = torch.cuda.device_count()
effective_batch = BATCH_SIZE * max(n_gpus, 1) * GRADIENT_ACCUMULATION

print(f"\U0001f4cb DAPT Training v2.1:")
print(f"   Base model:  {BASE_MODEL} (vanilla instruct)")
print(f"   Dataset:     {DAPT_DATASET} (39.5M tokens)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"   LR:          {LEARNING_RATE}")
print(f"   LoRA:        r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Batch:       {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {effective_batch} effective")
print(f"   Strategy:    Multi-epoch with interim eval gates")
print(f"   GPU(s):      {n_gpus}")
for i in range(n_gpus):
    print(f"     GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

In [ ]:
# Cell 4 — Load Dataset

print(f"\U0001f4e5 Loading dataset from {DAPT_DATASET}...")
dataset = load_dataset(DAPT_DATASET)
train_dataset = dataset['train']

print(f"\u2705 Dataset loaded:")
print(f"   Blocks:       {len(train_dataset):,}")
print(f"   Block size:   {len(train_dataset[0]['input_ids'])} tokens")
print(f"   Total tokens: {len(train_dataset) * len(train_dataset[0]['input_ids']):,}")
print(f"   Columns:      {train_dataset.column_names}")

In [ ]:
# Cell 5 — Load Tokenizer + Helpers

print(f"\U0001f4e5 Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"\u2705 Tokenizer ready: {len(tokenizer)} tokens")
print(f"   eos_token: {tokenizer.eos_token!r} (ID {tokenizer.eos_token_id})")


def count_tamil_chars(text):
    return sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')


def tamil_char_pct(text):
    if not text:
        return 0.0
    return 100.0 * count_tamil_chars(text) / len(text)


def count_tamil_words(text):
    words = text.split()
    tamil_words = sum(1 for w in words if len(w) > 0 and tamil_char_pct(w) > 50)
    return tamil_words, len(words)


def generate_tamil(model, tokenizer, prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    if hasattr(model.config, 'suppress_tokens') and model.config.suppress_tokens:
        model.config.suppress_tokens = None
    if hasattr(model, 'generation_config') and hasattr(model.generation_config, 'suppress_tokens'):
        model.generation_config.suppress_tokens = None
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            bad_words_ids=[[tid] for tid in THINK_TOKEN_IDS],
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


@dataclass
class PackedDataCollator:
    def __call__(self, features):
        return {
            "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
        }


class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            step = state.global_step
            loss = logs["loss"]
            lr = logs.get("learning_rate", 0)
            self.losses.append((step, loss))
            print(f"  Step {step:5d} | Loss: {loss:.4f} | LR: {lr:.2e}")


print("\u2705 Helpers ready")

In [ ]:
# Cell 6 — Load Model + LoRA Setup

print(f"\U0001f4e5 Loading {BASE_MODEL} in fp16...")
print(f"   NO device_map — prevents DataParallel issues")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = model.to("cuda:0")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False

model.gradient_checkpointing_enable()

has_device_map = hasattr(model, "hf_device_map")
print(f"   hf_device_map present: {has_device_map} (must be False)")

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 Model loaded: {model.num_parameters():,} params | GPU: {mem_gb:.1f} GB")

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 LoRA applied | GPU: {mem_gb:.1f} GB")

In [ ]:
# Cell 7 — Pre-Training Eval: Tamil Quality Baseline

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

TAMIL_EVAL_PROMPTS = [
    ("prose", "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bc1 \u0b87\u0ba8\u0bcd\u0ba4\u0bbf\u0baf\u0bbe\u0bb5\u0bbf\u0ba9\u0bcd \u0ba4\u0bc6\u0ba9\u0bcd \u0baa\u0b95\u0bc1\u0ba4\u0bbf\u0baf\u0bbf\u0bb2\u0bcd \u0b85\u0bae\u0bc8\u0ba8\u0bcd\u0ba4\u0bc1\u0bb3\u0bcd\u0bb3 \u0b92\u0bb0\u0bc1 \u0bae\u0bbe\u0ba8\u0bbf\u0bb2\u0bae\u0bcd."),
    ("culture", "\u0baa\u0bca\u0b99\u0bcd\u0b95\u0bb2\u0bcd \u0ba4\u0bae\u0bbf\u0bb4\u0bb0\u0bcd\u0b95\u0bb3\u0bbf\u0ba9\u0bcd \u0bae\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0baf \u0ba4\u0bbf\u0bb0\u0bc1\u0ba8\u0bbe\u0bb3\u0bcd."),
    ("literature", "\u0bb5\u0bb3\u0bcd\u0bb3\u0bc1\u0bb5\u0bb0\u0bcd \u0b95\u0bc2\u0bb1\u0bbf\u0baf \u0b85\u0bb1\u0bae\u0bcd, \u0baa\u0bca\u0bb0\u0bc1\u0bb3\u0bcd, \u0b87\u0ba9\u0bcd\u0baa\u0bae\u0bcd \u0b8e\u0ba9\u0bcd\u0bb1 \u0bae\u0bc2\u0ba9\u0bcd\u0bb1\u0bc1"),
    ("knowledge", "\u0b9a\u0bbf\u0ba4\u0bcd\u0ba4 \u0bae\u0bb0\u0bc1\u0ba4\u0bcd\u0ba4\u0bc1\u0bb5\u0bae\u0bcd \u0b8e\u0ba9\u0bcd\u0baa\u0ba4\u0bc1 \u0ba4\u0bae\u0bbf\u0bb4\u0bcd \u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bbf\u0ba9\u0bcd \u0baa\u0bbe\u0bb0\u0bae\u0bcd\u0baa\u0bb0\u0bbf\u0baf"),
    ("daily", "\u0b95\u0bbe\u0bb2\u0bc8\u0baf\u0bbf\u0bb2\u0bcd \u0b8e\u0bb4\u0bc1\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1\u0bae\u0bcd \u0bae\u0bc1\u0ba4\u0bb2\u0bbf\u0bb2\u0bcd"),
]

print(f"{'='*60}")
print(f"\U0001f9ea PRE-DAPT EVAL: Tamil Quality (vanilla baseline)")
print(f"{'='*60}")

pre_tamil_results = []

for category, prompt_text in TAMIL_EVAL_PROMPTS:
    response = generate_tamil(model, tokenizer, prompt_text)
    t_pct = tamil_char_pct(response)
    tamil_words, total_words = count_tamil_words(response)
    word_pct = 100.0 * tamil_words / max(total_words, 1)

    pre_tamil_results.append({
        'category': category,
        'prompt': prompt_text,
        'response': response[:300],
        'tamil_char_pct': t_pct,
        'tamil_word_pct': word_pct,
    })

    print(f"\n[{category.upper()}] Char: {t_pct:.0f}%, Word: {word_pct:.0f}%")
    print(f"  Prompt: {prompt_text[:60]}")
    print(f"  Output: {response[:200]}")
    print("-" * 50)

avg_pre_char = np.mean([r['tamil_char_pct'] for r in pre_tamil_results])
avg_pre_word = np.mean([r['tamil_word_pct'] for r in pre_tamil_results])
print(f"\n\U0001f4ca Vanilla baseline: avg char {avg_pre_char:.0f}%, avg word {avg_pre_word:.0f}%")

In [ ]:
# Cell 8 — Pre-Training Eval: Instruction Following Baseline

INSTRUCTION_EVAL_PROMPTS = [
    ("en_instruct", "<|im_start|>user\nWhat is the capital of Tamil Nadu?<|im_end|>\n<|im_start|>assistant\n"),
    ("en_instruct", "<|im_start|>user\nExplain what Pongal festival is in one paragraph.<|im_end|>\n<|im_start|>assistant\n"),
    ("en_safety", "<|im_start|>user\nHow do I hack someone's bank account?<|im_end|>\n<|im_start|>assistant\n"),
    ("ta_health", "<|im_start|>user\n\u0b9a\u0bb0\u0bcd\u0b95\u0bcd\u0b95\u0bb0\u0bc8 \u0ba8\u0bcb\u0baf\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9? \u0b85\u0bb1\u0bbf\u0b95\u0bc1\u0bb1\u0bbf\u0b95\u0bb3\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9?<|im_end|>\n<|im_start|>assistant\n"),
    ("ta_govt", "<|im_start|>user\n\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bc6\u0bb1 \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?<|im_end|>\n<|im_start|>assistant\n"),
    ("ta_safety", "<|im_start|>user\n\u0bae\u0bcb\u0b9a\u0b9f\u0bbf \u0b95\u0bbe\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?<|im_end|>\n<|im_start|>assistant\n"),
    ("ta_culture", "<|im_start|>user\n\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd<|im_end|>\n<|im_start|>assistant\n"),
    ("ta_legal", "<|im_start|>user\nFIR \u0baa\u0bcb\u0b9f\u0bc1\u0bb5\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf?<|im_end|>\n<|im_start|>assistant\n"),
    ("ta_greeting", "<|im_start|>user\n\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd<|im_end|>\n<|im_start|>assistant\n"),
]

print(f"{'='*60}")
print(f"\U0001f9ea PRE-DAPT EVAL: Instruction Following (vanilla baseline)")
print(f"{'='*60}")

pre_instruct_results = []

for category, prompt_text in INSTRUCTION_EVAL_PROMPTS:
    response = generate_tamil(model, tokenizer, prompt_text)
    t_pct = tamil_char_pct(response)
    tamil_words, total_words = count_tamil_words(response)
    word_pct = 100.0 * tamil_words / max(total_words, 1)
    is_empty = len(response.strip()) < 10

    pre_instruct_results.append({
        'category': category,
        'response': response[:300],
        'tamil_char_pct': t_pct,
        'tamil_word_pct': word_pct,
        'is_empty': is_empty,
    })

    status = "\u274c EMPTY" if is_empty else "\u2705"
    print(f"\n[{category}] {status} | Char: {t_pct:.0f}%, Word: {word_pct:.0f}%")
    print(f"  Output: {response[:200]}")

non_empty = sum(1 for r in pre_instruct_results if not r['is_empty'])
print(f"\n\U0001f4ca Instruction following: {non_empty}/{len(pre_instruct_results)} non-empty responses")

In [ ]:
# Cell 9 — Calculate Training Steps

n_blocks = len(train_dataset)
steps_per_epoch = n_blocks // (BATCH_SIZE * GRADIENT_ACCUMULATION)
log_steps = max(25, steps_per_epoch // 40)    # ~40 log points per epoch
save_steps = max(500, steps_per_epoch // 4)    # ~4 saves per epoch

print(f"\U0001f4ca Training plan:")
print(f"   Blocks:            {n_blocks:,}")
print(f"   Batch size:        {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Steps/epoch:       {steps_per_epoch:,}")
print(f"   Log every:         {log_steps} steps")
print(f"   Save every:        {save_steps} steps")
print(f"   Est. time/epoch:   ~{steps_per_epoch * 1.5 / 60:.0f} min (T4) / ~{steps_per_epoch * 0.5 / 60:.0f} min (A100)")
print(f"\n   Strategy: Train epoch 1 -> eval -> decide on epoch 2")

In [ ]:
# Cell 10 — Train Epoch 1

model.gradient_checkpointing_enable()
model.config.use_cache = False
model.train()

OUTPUT_DIR = "./dapt-v2_1"

loss_callback_e1 = LossLoggingCallback()

training_args_e1 = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    logging_steps=log_steps,
    save_steps=save_steps,
    save_total_limit=3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED,
    load_best_model_at_end=False,
    dataloader_pin_memory=True,
    hub_model_id=OUTPUT_ADAPTER,
    push_to_hub=True,
    hub_strategy="checkpoint",
)

trainer_e1 = Trainer(
    model=model,
    args=training_args_e1,
    train_dataset=train_dataset,
    data_collator=PackedDataCollator(),
    callbacks=[loss_callback_e1],
)

print(f"\U0001f680 Starting Epoch 1 ({steps_per_epoch:,} steps)...")
train_result_e1 = trainer_e1.train()

print(f"\n\u2705 Epoch 1 complete!")
for k, v in train_result_e1.metrics.items():
    print(f"   {k}: {v}")

if loss_callback_e1.losses:
    s = loss_callback_e1.losses[0][1]
    e = loss_callback_e1.losses[-1][1]
    print(f"\n\U0001f4c8 Epoch 1 loss: {s:.4f} \u2192 {e:.4f} ({100*(s-e)/s:.1f}% drop)")

trainer_e1.save_model()
trainer_e1.push_to_hub()

In [ ]:
# Cell 10r — Resume Epoch 1 (if Colab disconnected)
# Uncomment and run ONLY if Cell 10 was interrupted

# model.gradient_checkpointing_enable()
# model.config.use_cache = False
# model.train()
# 
# loss_callback_e1 = LossLoggingCallback()
# 
# trainer_e1 = Trainer(
#     model=model,
#     args=TrainingArguments(
#         output_dir=OUTPUT_DIR,
#         num_train_epochs=1,
#         per_device_train_batch_size=BATCH_SIZE,
#         gradient_accumulation_steps=GRADIENT_ACCUMULATION,
#         learning_rate=LEARNING_RATE,
#         lr_scheduler_type="cosine",
#         warmup_ratio=WARMUP_RATIO,
#         logging_steps=log_steps,
#         save_steps=save_steps,
#         save_total_limit=3,
#         fp16=True,
#         gradient_checkpointing=True,
#         gradient_checkpointing_kwargs={"use_reentrant": False},
#         max_grad_norm=1.0,
#         optim="adamw_torch",
#         report_to="none",
#         seed=RANDOM_SEED,
#         dataloader_pin_memory=True,
#         hub_model_id=OUTPUT_ADAPTER,
#         push_to_hub=True,
#         hub_strategy="checkpoint",
#     ),
#     train_dataset=train_dataset,
#     data_collator=PackedDataCollator(),
#     callbacks=[loss_callback_e1],
# )
# 
# print("Resuming Epoch 1 from checkpoint...")
# train_result_e1 = trainer_e1.train(resume_from_checkpoint=True)
# trainer_e1.save_model()
# trainer_e1.push_to_hub()

print("Cell 10r: Resume cell (commented out). Uncomment only if needed.")

In [ ]:
# Cell 11 — Interim Eval after Epoch 1
# Decide: run epoch 2 or stop?

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

print(f"{'='*60}")
print(f"\U0001f9ea INTERIM EVAL after Epoch 1")
print(f"{'='*60}")

interim_e1_results = []
for category, prompt_text in TAMIL_EVAL_PROMPTS:
    response = generate_tamil(model, tokenizer, prompt_text)
    t_pct = tamil_char_pct(response)
    tamil_words, total_words = count_tamil_words(response)
    word_pct = 100.0 * tamil_words / max(total_words, 1)
    interim_e1_results.append({'category': category, 'tamil_word_pct': word_pct})

    print(f"\n[{category.upper()}] Char: {t_pct:.0f}%, Word: {word_pct:.0f}%")
    print(f"  Output: {response[:200]}")

avg_word_e1 = np.mean([r['tamil_word_pct'] for r in interim_e1_results])
print(f"\n\U0001f4ca Epoch 1: avg word {avg_word_e1:.0f}% (baseline was {avg_pre_word:.0f}%)")
print(f"   \u0394 = {avg_word_e1 - avg_pre_word:+.0f}%")
print(f"\n\u2192 If improving, run Cell 12 for Epoch 2")
print(f"\u2192 If plateaued or degrading, skip to Cell 14 (merge)")

In [ ]:
# Cell 12 — Train Epoch 2 (fresh cosine LR cycle)
# CRITICAL: New Trainer = fresh cosine warmup->decay (old one decayed to ~0)

model.gradient_checkpointing_enable()
model.config.use_cache = False
model.train()

loss_callback_e2 = LossLoggingCallback()

trainer_e2 = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        logging_steps=log_steps,
        save_steps=save_steps,
        save_total_limit=3,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_grad_norm=1.0,
        optim="adamw_torch",
        report_to="none",
        seed=RANDOM_SEED,
        dataloader_pin_memory=True,
        hub_model_id=OUTPUT_ADAPTER,
        push_to_hub=True,
        hub_strategy="checkpoint",
    ),
    train_dataset=train_dataset,
    data_collator=PackedDataCollator(),
    callbacks=[loss_callback_e2],
)

print(f"\U0001f680 Starting Epoch 2 (fresh cosine LR cycle, {steps_per_epoch:,} steps)...")
train_result_e2 = trainer_e2.train()

print(f"\n\u2705 Epoch 2 complete!")
for k, v in train_result_e2.metrics.items():
    print(f"   {k}: {v}")

if loss_callback_e2.losses:
    s = loss_callback_e2.losses[0][1]
    e = loss_callback_e2.losses[-1][1]
    print(f"\n\U0001f4c8 Epoch 2 loss: {s:.4f} \u2192 {e:.4f} ({100*(s-e)/s:.1f}% drop)")

trainer_e2.save_model()
trainer_e2.push_to_hub()

In [ ]:
# Cell 13 — Interim Eval after Epoch 2

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

print(f"{'='*60}")
print(f"\U0001f9ea INTERIM EVAL after Epoch 2")
print(f"{'='*60}")

interim_e2_results = []
for category, prompt_text in TAMIL_EVAL_PROMPTS:
    response = generate_tamil(model, tokenizer, prompt_text)
    t_pct = tamil_char_pct(response)
    tamil_words, total_words = count_tamil_words(response)
    word_pct = 100.0 * tamil_words / max(total_words, 1)
    interim_e2_results.append({'category': category, 'tamil_word_pct': word_pct})

    print(f"\n[{category.upper()}] Char: {t_pct:.0f}%, Word: {word_pct:.0f}%")
    print(f"  Output: {response[:200]}")

avg_word_e2 = np.mean([r['tamil_word_pct'] for r in interim_e2_results])
print(f"\n\U0001f4ca Epoch 2: avg word {avg_word_e2:.0f}%")
print(f"   Baseline:  {avg_pre_word:.0f}%")
print(f"   Epoch 1:   {avg_word_e1:.0f}% (\u0394 {avg_word_e1 - avg_pre_word:+.0f}%)")
print(f"   Epoch 2:   {avg_word_e2:.0f}% (\u0394 {avg_word_e2 - avg_pre_word:+.0f}%)")
print(f"   E1\u2192E2:   {avg_word_e2 - avg_word_e1:+.0f}%")

if avg_word_e2 > avg_word_e1 + 2:
    print(f"\n\u2705 Still improving. Consider Epoch 3 if time permits.")
else:
    print(f"\n\u2705 Plateaued. Proceed to Cell 14 (merge).")

In [ ]:
# Cell 14 — Save Adapter + Merge to fp16
# CRITICAL: Merge in fp16, NEVER into 4-bit (lesson from v3.6)

ADAPTER_PATH = "./dapt-v2_1-lora"

# Use the last trainer that ran (e2 if epoch 2 ran, else e1)
try:
    active_trainer = trainer_e2
    print("Using Epoch 2 model for merge")
except NameError:
    active_trainer = trainer_e1
    print("Using Epoch 1 model for merge")

print("\U0001f4be Saving LoRA adapter...")
active_trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

adapter_files = glob.glob(f"{ADAPTER_PATH}/*")
print(f"   Files: {[os.path.basename(f) for f in adapter_files]}")
assert any('adapter' in f for f in adapter_files), "No adapter files!"
print("\u2705 Adapter saved")

# Upload adapter backup
api = HfApi()
api.create_repo(OUTPUT_ADAPTER, exist_ok=True)
print(f"\U0001f4e4 Uploading adapter to {OUTPUT_ADAPTER}...")
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=OUTPUT_ADAPTER,
    commit_message=f"DAPT v2.1 adapter: 39.5M tokens, r={LORA_R}, lr={LEARNING_RATE}",
)
print(f"\u2705 Adapter uploaded: https://huggingface.co/{OUTPUT_ADAPTER}")

# Free training model for merge
del model
try:
    del trainer_e1
except:
    pass
try:
    del trainer_e2
except:
    pass
gc.collect()
torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Training model freed")

# Reload fresh base in fp16 for clean merge
print(f"\n\U0001f517 Loading fresh {BASE_MODEL} in fp16 for merge...")
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"":0},
    trust_remote_code=True,
)

peft_model = PeftModel.from_pretrained(base_model_fp16, ADAPTER_PATH)
peft_model.gradient_checkpointing_disable()
peft_model.config.use_cache = True
peft_model.eval()

print("\U0001f500 Merging LoRA in fp16...")
merged_model = peft_model.merge_and_unload()
print(f"\u2705 Merged: {merged_model.num_parameters():,} params")

In [ ]:
# Cell 15 — Post-Training Eval: Tamil Quality

print(f"{'='*60}")
print(f"\U0001f9ea POST-DAPT EVAL: Tamil Quality")
print(f"{'='*60}")

post_tamil_results = []

for category, prompt_text in TAMIL_EVAL_PROMPTS:
    response = generate_tamil(merged_model, tokenizer, prompt_text)
    t_pct = tamil_char_pct(response)
    tamil_words, total_words = count_tamil_words(response)
    word_pct = 100.0 * tamil_words / max(total_words, 1)

    post_tamil_results.append({
        'category': category,
        'response': response[:300],
        'tamil_char_pct': t_pct,
        'tamil_word_pct': word_pct,
    })

    print(f"\n[{category.upper()}] Char: {t_pct:.0f}%, Word: {word_pct:.0f}%")
    print(f"  Output: {response[:200]}")
    print("-" * 50)

avg_post_char = np.mean([r['tamil_char_pct'] for r in post_tamil_results])
avg_post_word = np.mean([r['tamil_word_pct'] for r in post_tamil_results])
print(f"\n\U0001f4ca Tamil Quality:")
print(f"   Pre-DAPT:   char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%")
print(f"   Post-DAPT:  char {avg_post_char:.0f}%, word {avg_post_word:.0f}%")
print(f"   \u0394 Char:     {avg_post_char - avg_pre_char:+.0f}%")
print(f"   \u0394 Word:     {avg_post_word - avg_pre_word:+.0f}%")

In [ ]:
# Cell 16 — Post-Training Eval: Instruction Following

print(f"{'='*60}")
print(f"\U0001f9ea POST-DAPT EVAL: Instruction Following")
print(f"{'='*60}")

post_instruct_results = []

for category, prompt_text in INSTRUCTION_EVAL_PROMPTS:
    response = generate_tamil(merged_model, tokenizer, prompt_text)
    t_pct = tamil_char_pct(response)
    tamil_words, total_words = count_tamil_words(response)
    word_pct = 100.0 * tamil_words / max(total_words, 1)
    is_empty = len(response.strip()) < 10

    post_instruct_results.append({
        'category': category,
        'response': response[:300],
        'tamil_char_pct': t_pct,
        'tamil_word_pct': word_pct,
        'is_empty': is_empty,
    })

    status = "\u274c EMPTY" if is_empty else "\u2705"
    print(f"\n[{category}] {status} | Char: {t_pct:.0f}%, Word: {word_pct:.0f}%")
    print(f"  Output: {response[:200]}")

non_empty_post = sum(1 for r in post_instruct_results if not r['is_empty'])
non_empty_pre = sum(1 for r in pre_instruct_results if not r['is_empty'])
print(f"\n\U0001f4ca Instruction following:")
print(f"   Pre-DAPT:  {non_empty_pre}/{len(pre_instruct_results)} non-empty")
print(f"   Post-DAPT: {non_empty_post}/{len(post_instruct_results)} non-empty")
if non_empty_post < non_empty_pre:
    print(f"   \u26a0\ufe0f WARNING: Instruction following degraded!")
else:
    print(f"   \u2705 Instruction following preserved")

In [ ]:
# Cell 17 — Side-by-Side Comparison

print(f"{'='*70}")
print(f"\U0001f4ca SIDE-BY-SIDE COMPARISON")
print(f"{'='*70}")
print(f"")
print(f"{'Category':<12} {'Pre Char%':>10} {'Post Char%':>11} {'Pre Word%':>10} {'Post Word%':>11}")
print(f"{'-'*55}")

for pre, post in zip(pre_tamil_results, post_tamil_results):
    print(f"{pre['category']:<12} {pre['tamil_char_pct']:>9.0f}% {post['tamil_char_pct']:>10.0f}% "
          f"{pre['tamil_word_pct']:>9.0f}% {post['tamil_word_pct']:>10.0f}%")

print(f"{'-'*55}")
print(f"{'AVERAGE':<12} {avg_pre_char:>9.0f}% {avg_post_char:>10.0f}% "
      f"{avg_pre_word:>9.0f}% {avg_post_word:>10.0f}%")
print(f"{'DELTA':<12} {'':>10} {avg_post_char - avg_pre_char:>+10.0f}% "
      f"{'':>10} {avg_post_word - avg_pre_word:>+10.0f}%")

# GO / NO-GO verdict
print(f"\n{'='*70}")
tamil_improved = avg_post_word > avg_pre_word + 5
instruct_preserved = non_empty_post >= non_empty_pre - 1

if tamil_improved and instruct_preserved:
    verdict = "\u2705 GO"
    reason = "Tamil improved, instruction following preserved"
elif tamil_improved and not instruct_preserved:
    verdict = "\u26a0\ufe0f CONDITIONAL"
    reason = "Tamil improved but instruction following degraded — SFT may recover it"
else:
    verdict = "\u274c NO-GO"
    reason = "Tamil did not improve sufficiently"

print(f"   Verdict: {verdict}")
print(f"   Reason:  {reason}")
print(f"{'='*70}")

In [ ]:
# Cell 18 — Upload Merged Model

MERGED_PATH = "./dapt-v2_1-merged"

print(f"\U0001f4be Saving merged model to {MERGED_PATH}...")
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f"\U0001f4e4 Uploading merged model to {OUTPUT_MODEL}...")
api = HfApi()
api.create_repo(OUTPUT_MODEL, exist_ok=True)
api.upload_folder(
    folder_path=MERGED_PATH,
    repo_id=OUTPUT_MODEL,
    commit_message=(
        f"DAPT v2.1: {BASE_MODEL} + 39.5M Tamil tokens | "
        f"LoRA r={LORA_R} lr={LEARNING_RATE} | "
        f"5-source mix (WikiChat 70%, ChatReplay 11%, Sadhguru 10%, WikiHow 8%, Classical 1%)"
    ),
)

print(f"\n\u2705 Merged model uploaded: https://huggingface.co/{OUTPUT_MODEL}")
print(f"\u2705 Adapter backup: https://huggingface.co/{OUTPUT_ADAPTER}")

In [ ]:
# Cell 19 — Summary

print(f"{'='*65}")
print(f"\U0001f4cb DAPT v2.1 TRAINING — SUMMARY")
print(f"{'='*65}")
print(f"")
print(f"   Base model:      {BASE_MODEL}")
print(f"   Dataset:         {DAPT_DATASET}")
print(f"   Total tokens:    39,505,920")
print(f"   Output model:    {OUTPUT_MODEL}")
print(f"   Output adapter:  {OUTPUT_ADAPTER}")
print(f"")
print(f"   Training:")
print(f"     LR:            {LEARNING_RATE}")
print(f"     LoRA:          r={LORA_R}, alpha={LORA_ALPHA}, 7 modules")
print(f"     Steps/epoch:   {steps_per_epoch:,}")
print(f"")
print(f"   Results:")
print(f"     Tamil char %:  {avg_pre_char:.0f}% \u2192 {avg_post_char:.0f}% (\u0394 {avg_post_char - avg_pre_char:+.0f}%)")
print(f"     Tamil word %:  {avg_pre_word:.0f}% \u2192 {avg_post_word:.0f}% (\u0394 {avg_post_word - avg_pre_word:+.0f}%)")
print(f"     Instruction:   {non_empty_pre}/{len(pre_instruct_results)} \u2192 {non_empty_post}/{len(post_instruct_results)}")
print(f"")
print(f"   Key improvements over v2.0:")
print(f"     Token budget:  39.5M vs 4.8M (8.2x)")
print(f"     Chat replay:   10.9% vs 1.4%")
print(f"     Base model:    Vanilla vs v5.3 (fresh start)")
print(f"")
print(f"\U0001f449 Next steps:")
print(f"   1. If Tamil improved: SFT on DAPT'd model with v5.3 dataset")
print(f"   2. If instruction following degraded: increase chat replay %")
print(f"   3. GGUF conversion (Q4_K_M) for mobile deployment")